# Demo notebook to run a pre-trained model from ML4Fires on CMIP data

First import relevant libraries and load required configuration files and masks for data

In [ ]:
import numpy as np
import xarray as xr
import toml
import munch
import ipywidgets as widgets

import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append('../')
from Fires._utilities.utils_mlflow import load_model_from_mlflow
from Fires._utilities.utils_inference import get_cmip6_inference, process_and_plot_cmip6infer, process_and_plot_data, load_input_data

In [ ]:
config = munch.munchify(toml.load("../config/cmip6_inference.toml"))
searfire_ds_path = "../../ML4Fires_data/data_100km.zarr"
sea_poles_mask_path = "../data/land_sea_poles_mask.nc"
seafire_ds = xr.open_zarr(searfire_ds_path)
sea_poles_mask = xr.open_dataset(sea_poles_mask_path)

Select a climate scenario (ssp), climate model and time range (years) from the list. The data used in this demo refers to ScenarioMIP data from CMIP6 (Coupled Model Intercomparison Project).

In [ ]:
scenario = widgets.Dropdown(
    options=[('SSP126', 'ssp126'), ('SSP245', 'ssp245'), ('SSP370', 'ssp370'),
             ('SSP585', 'ssp585')],
    value = 'ssp126',
    style={'description_width': '60px'}, 
    description='Scenario', disabled=False,
    layout=widgets.Layout(width='200px'))

climate_model = widgets.Dropdown(
    options=[('MPI-ESM1-2-HR', 'MPI-ESM1-2-HR'), ('CMCC-ESM2', 'CMCC-ESM2'), ('NorESM2-MM', 'NorESM2-MM'),
             ('CESM2', 'CESM2')],
    value = 'CMCC-ESM2',
    style={'description_width': '60px'}, 
    description='Model', disabled=False,
    layout=widgets.Layout(width='200px'))

year_range = widgets.IntRangeSlider(
    value=[2030, 2035],        # initial range
    min=2015,                 # min value
    max=2100,               # max value
    step=1,                # step size
    description='Year range:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

display(widgets.HBox([scenario, climate_model, year_range]))

In [ ]:
assert scenario.value != None, "Please select a CMIP6 scenario in the previous cell before proceesing"

Download a pre-trained model from MLflow. The provenance document will also be downloaded and can be used to trace the process followed for training the model. More details on the default model implementation are available at: 

[ML4Fires/docs/the_backbone_architecture.md](../docs/the_backbone_architecture.md)

In [ ]:
run_name=input()
registered_model = load_model_from_mlflow(run_name, provenance=True)

Generate the burned areas maps on the selected CMIP data with the pre-trained model. This step will:

1. load the variables from the selected climate model. Some info about the a default set variables used for training is reported at [ML4Fires/data/the_backbone_architecture.md](../docs/data.md)
2. preprocess the variables according to the configuration file defined in `config`. Default configuration is defined in [ML4Fires/config/cmip6_inference.toml](../config/cmip6_inference.toml)
3. run the pre-trained model on the climate data
4. return burned area maps predictions

In [ ]:
predictions = get_cmip6_inference(
    seafire_ds=seafire_ds,
    run_name=run_name,
    scenario=scenario,
    climate_model=climate_model,
    year_range=year_range,
    infer_config=config,
    model=registered_model)

Aggregate the burned areas maps on monthly, yearly or decadal time scales

In [ ]:
monthly_agg = widgets.Dropdown(
    options=[('Mean', 'mean'), ('Sum', 'sum'), ('Median', 'median'),
             ('Mode', 'mode')],
    value = 'sum',
    style={'description_width': '150px'}, 
    description='Monthly aggregate', disabled=False,)

monthly_add_period = widgets.Dropdown(
    options=[('1 Month', '1M'), ('2 Months', '2M'), ('4 Months', '4M'),
             ('6 Month', '6M')],
    value = '1M',
    style={'description_width': '150px'}, 
    description='Monthly aggregate period', disabled=False,)

yearly_agg = widgets.Dropdown(
    options=[('Mean', 'mean'), ('Sum', 'sum'), ('Median', 'median'),
             ('Mode', 'mode')],
    value = 'mean',
    style={'description_width': '150px'}, 
    description='Yearly aggregate', disabled=False,)

decadal_agg = widgets.Dropdown(
    options=[('Mean', 'mean'), ('Sum', 'sum'), ('Median', 'median'),
             ('Mode', 'mode')],
    value = 'mean',
    style={'description_width': '150px'}, 
    description='Decadal aggregate', disabled=False,)

display(widgets.HBox([monthly_agg,
                      monthly_add_period]))
        
display(widgets.VBox([yearly_agg,
                      decadal_agg]))


Plot the resulting aggregated maps, showing the extent of burned area per grid cell (1° x 1°)

In [ ]:
input_data = load_input_data(searfire_ds_path, '2019', '2020') # Required by internal setting of variables in Fires._utilities.utils_inference

temporal_aggregate_scheme = {"monthly":[monthly_agg.value,monthly_add_period.value],
                             "yearly":yearly_agg.value,
                            "decadal":decadal_agg.value}
process_and_plot_cmip6infer(
	data=predictions.global_burned_areas,
	temporal_aggregate_scheme=temporal_aggregate_scheme,
	label=f'Averaged burned area for {year_range.value[0]}-{year_range.value[1]} in scenario {scenario.value}',
	lats=predictions.latitude.values,
	lons=predictions.longitude.values,
    scale_min=0,
    scale_max=8000,
	model_name="Unet ++",
	sea_poles_mask=sea_poles_mask["basis_regions"].sel(cl=0)
)